# Notebook 02: Getting data from EarthScope

**Goal (about 1 hour):** understand where the seismograms come from and request some yourself, then see why this repo caches them.

Nearly all permanent-network seismic data in the US are archived by EarthScope (formerly IRIS DMC) and served through standard **FDSN web services**. ObsPy wraps them in one client. Read along with:

- ObsPy's [FDSN client documentation](https://docs.obspy.org/packages/obspy.clients.fdsn.html) and the tutorial page [Retrieving Data from Data Centers](https://docs.obspy.org/tutorial/code_snippets/retrieving_data_from_datacenters.html).
- Seismo-Live's [06 FDSN](https://seismo-live.github.io/html/ObsPy/06_FDSN_wrapper.html) notebook (Krischer, CC BY-NC-SA 4.0).
- EarthScope's service index at [service.earthscope.org](https://service.earthscope.org/). Data use falls under their terms of service; cite per [earthscope.org/how-to-cite](https://www.earthscope.org/how-to-cite/).

In [1]:
# --- setup: make the repo importable and keep figures inline ---
import sys, warnings
from pathlib import Path
REPO = Path.cwd().parent if (Path.cwd() / "../rainier_sws").exists() else Path.cwd()
sys.path.insert(0, str(REPO))
warnings.filterwarnings("ignore")
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
from rainier_sws import paths, data, measure, quality, stats
print("repo:", REPO)

repo: /home/claude/rainier-sws-tutorials


## 1. The three questions a data request answers

Every request names **who** (network and station), **what** (channel and location code), and **when** (start and end time). The vocabulary:

| field | example | meaning |
| --- | --- | --- |
| network | `UW` | the operator (UW = Pacific Northwest Seismic Network); `CC` = Cascades Volcano Observatory |
| station | `STAR` | the site |
| location | `01` or `--` | distinguishes several instruments at one site; wildcard `*` is safest |
| channel | `EHZ` or `?H?` | band code, instrument code, orientation; `?H?` matches all seismic channels |

## 2. Station metadata

Before asking for waveforms, ask what exists. `get_stations` returns an **Inventory** (network > station > channel). This is a live request; if EarthScope is unreachable from your machine the cell reports that and we fall back to the station table shipped with the repo.

In [2]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
ONLINE = True
try:
    client = Client(data.FDSN_URL, timeout=30)
    inv = client.get_stations(network="UW", station="STAR", level="channel",
                              starttime=UTCDateTime("2025-07-01"), endtime=UTCDateTime("2025-08-01"))
    print(inv)
except Exception as exc:
    ONLINE = False
    print("Offline (", type(exc).__name__, "). Using the repo's station table instead:")
    print(data.load_stations().query("station == 'STAR'"))

Offline ( FDSNNoServiceException ). Using the repo's station table instead:
   network station        lat         lon  elevation
30      UW    STAR  46.850849 -121.792953     3365.0


## 3. Requesting waveforms

The research pipeline requests `[origin - 5 s, origin + 15 s]` for each earthquake. One such request looks like this. (It runs only if you are online; otherwise the next section shows the cached copy of the same window.)

In [3]:
cat = data.load_station_catalog("STAR")
row = cat[cat.evid == 61504668].iloc[0]
t0 = data.to_utc(row.start_time); t1 = t0 + data.PRE_S + data.POST_S
print("request window:", t0, "->", t1)
if ONLINE:
    st_live = data.fetch_from_earthscope("UW", "STAR", t0, t1)
    print(st_live)

request window: 2025-07-18T14:09:59.070000Z -> 2025-07-18T14:10:19.070000Z


## 4. Why we cache: replicability and politeness

A request takes a second or two and depends on a server being up. Doing it once per earthquake per station per run is slow, fragile, and means two runs may see different data (archives get backfilled and corrected). So the helper `data.get_event_stream`:

1. looks in `data/waveforms/<STATION>/` for a trace starting at the requested time;
2. only if it is missing, asks EarthScope and saves the answer to the cache.

Look at the function now (`rainier_sws/data.py`). Then use it.

In [4]:
ev = data.get_event_stream("STAR", row)   # cache first, EarthScope second
print("source:", ev.source); print(ev)
if ONLINE:
    # same bytes? (a replicability check)
    for tr_c in ev:
        tr_l = st_live.select(channel=tr_c.stats.channel)[0]
        n = min(len(tr_c), len(tr_l))
        print(tr_c.stats.channel, "identical samples:", np.array_equal(tr_c.data[:n], tr_l.data[:n]))

source: cache
3 Trace(s) in Stream:
UW.STAR.01.EHZ | 2025-07-18T14:09:59.070000Z - 2025-07-18T14:10:19.060000Z | 100.0 Hz, 2000 samples
UW.STAR.01.EHN | 2025-07-18T14:09:59.070000Z - 2025-07-18T14:10:19.060000Z | 100.0 Hz, 2000 samples
UW.STAR.01.EHE | 2025-07-18T14:09:59.070000Z - 2025-07-18T14:10:19.060000Z | 100.0 Hz, 2000 samples


## 5. Which catalog events are cached?

The STAR catalog has more rows than the cache has earthquakes (some windows were never downloaded or came back with fewer than 3 channels). This matters later: *the set of events you measure is itself a choice that affects the answer.*

In [5]:
cache = data.describe_cache("STAR")
print(cache.complete.value_counts().rename({True: "3 channels cached", False: "missing/incomplete"}))
missing = cache[~cache.complete].evid.tolist()
print("first few missing evids:", missing[:8])

complete
3 channels cached     901
missing/incomplete    295
Name: count, dtype: int64
first few missing evids: [61504463, 61504478, 61504488, 61504518, 61504528, 61504583, 61504623, 61504638]


## 6. Lessons the research repo learned the hard way

Read these now so you do not rediscover them (all from `mt-rainier/CLAUDE.md`):

- Request `?H?`, not `*`. Some old sites (LON) carry 40+ non-seismic channel codes; a `*` request can time out at the server.
- Use location `*`. STAR's location code is `01`; assuming blank silently returns nothing.
- `Client("IRIS")` and `Client("https://service.earthscope.org")` are the same archive.
- Some stations have metadata but no archived waveforms for 2025 (they exist only in real-time feeds). Metadata is not data.
- When a bulk request fails with a 504, it is usually the request, not the network.

## 7. Exercises

1. If online: fetch one of the `missing` events above with `data.get_event_stream("STAR", cat[cat.evid == missing[0]].iloc[0])`. Confirm it is now in the cache (`describe_cache` again). If offline, write down the command you would run.
2. Fetch (or look up in the station table) the coordinates of RCM and MILD. How far are they from STAR? (`obspy.geodetics.gps2dist_azimuth` returns distance in metres and azimuth.)
3. Using `client.get_events` (ObsPy docs) and if online, ask the USGS catalog for earthquakes within 20 km of the summit (46.853 N, 121.760 W) in July 2025. How many does it return compared with the PNSN export in `data/catalog/rainier_events.csv`? Different catalogs, different counts: another thing to be aware of.

Next: `03_earthquake_catalogs_and_picks.ipynb`.